# Notebook 12 — Multinomial Logistic Regression: Cluster Membership from Triage Variables

## Purpose

This notebook fits multinomial logistic regression models to predict cluster membership from variables available at the time of triage (before any resource consumption). The goal is to quantify which patient characteristics — demographics, vital signs, chief complaint, transport mode, and optionally triage score — are associated with belonging to each resource utilisation cluster relative to a clinical reference cluster.

---

## Cluster configuration
Both the 5-cluster (mcs=3407, ms=170) and 9-cluster (mcs=2271, ms=15) solutions are analysed. The reference cluster in each case is the largest, clinically least-intensive group (C5 — discharged, minimal consumption).

---

## Pipeline Overview

### 1. Feature preparation
Categorical variables are formatted for patsy formula encoding with explicit reference modalities set via `REF_CATEGORIES`: male sex, personal transport, age group 15–30, Trauma chief complaint, Triage 3, and clinical-norm status levels (normotension, normocardia, normothermia, etc.). Rare complaint categories (Metabolic, Hematology, Poisoning) are merged into a single `Metabolic_Hematologic_Toxic` group to avoid quasi-separation.

### 2. Spline analysis of age
A natural cubic spline (4 degrees of freedom) is fitted to age within a MNLogit framework to test non-linearity of the age–cluster relationship. For each non-reference cluster, a subplot shows the logit-scale spline curve with 95% CI, overlaid with empirical logit points per age bin. Results justify the use of categorical `age_group` in the main regression.

### 3. Multinomial logistic regression — two models
Two models are run per clustering solution:
- **With triage score** (`ALL_REG_FEATURES_WITH_TRIAGE`)
- **Without triage score** (`ALL_REG_FEATURES_NO_TRIAGE`) — simulating a scenario where triage level is not yet assigned

### 4. Results formatting and forest plots
The `format_mnlogit_results()` helper extracts odds ratios, 95% CIs, and p-values for all covariates across all cluster contrasts. Forest plots are generated with log-OR on the x-axis, significant results in crimson and non-significant in steel blue, one panel per cluster vs. reference. Interpretation focuses on ORs > 1.5 or < 0.67, given the large sample size (N ≈ 30,000) makes most associations statistically significant regardless of clinical relevance.

### 5. Binary logistic regression — outlier profile
A separate binary logistic regression models the probability of being an outlier (HDBSCAN label = −1) as a function of the same triage variables, to characterise patients whose resource use pattern is structurally atypical.

In [138]:
# ==============================================================================
# CHUNK 1 — Imports, load, format, modality exploration
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import matplotlib
matplotlib.rcdefaults()
plt.style.use('default')
plt.rcParams['text.color']        = 'black'
plt.rcParams['axes.labelcolor']   = 'black'
plt.rcParams['xtick.color']       = 'black'
plt.rcParams['ytick.color']       = 'black'
plt.rcParams['figure.facecolor']  = 'white'
plt.rcParams['axes.facecolor']    = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import mnlogit
from patsy import dmatrix, cr
import warnings
warnings.filterwarnings("ignore")
import re

# ── Paths ──────────────────────────────────────────────────────────────────────
RUN_LABEL = "s2_balanced"
SCALER    = "minmax"

BASE_DIR    = f"Results/Regular_clustering/Full_dataset/With_counts/{SCALER}/{RUN_LABEL}"


# ── Cluster labels mapping ─────────────────────────────────────────────────────
CLUSTERING_RUNS = [
    {
        'n_clusters'    : 5,
        'mcs'           : 3407,
        'ms'            : 170,
        'ref_cluster'   : 3,
        'cluster_labels': {
            1 : "C1 — UHCD + hospitalization + heavy workup",
            0 : "C2 — Hospitalized + full workup",
            2 : "C3 — Discharged + biology +/- ECG",
            4 : "C4 — Discharged + isolated X-ray +/- CT",
            3 : "C5 — Discharged + minimal consumption",
            -1: "Outliers",
        },
        'cluster_order' : [
            "C1 — UHCD + hospitalization + heavy workup",
            "C2 — Hospitalized + full workup",
            "C3 — Discharged + biology +/- ECG",
            "C4 — Discharged + isolated X-ray +/- CT",
            "C5 — Discharged + minimal consumption",
            "Outliers",
        ],
    },
    {
        'n_clusters'    : 9,
        'mcs'           : 2271,
        'ms'            : 15,
        'ref_cluster'   : 2,
        'cluster_labels': {
            0 : "C1 — UHCD + hospitalization + mixed workup ++",
            8 : "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            5 : "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            6 : "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            4 : "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            7 : "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            1 : "C7 — Discharged + biology + ECG+",
            3 : "C8 — Discharged + isolated X-ray",
            2 : "C9 — Discharged + minimal consumption",
            -1: "Outliers",
        },
        'cluster_order' : [
            "C1 — UHCD + hospitalization + mixed workup ++",
            "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            "C7 — Discharged + biology + ECG+",
            "C8 — Discharged + isolated X-ray",
            "C9 — Discharged + minimal consumption",
            "Outliers",
        ],
    },
]


# ── Feature lists ──────────────────────────────────────────────────────────────
''' group complaint too sparse n some clusters'''
# ── Create grouped complaint category ─────────────────────────────────────────
SYSTEMIC_RARE = [
    "Metabolic_Endocrine",
    "Hematology",
    "Poisoning_Intoxication",
]




REG_FEATURES_STATUS = [
    "bp_status",
    "hr_status",
    "temp_status",
    "sat_status",
    "rr_status",
    "o2_flow_status",
    "gcs_status",
    "cap_blood_sugar_status",
    "anisocoria_status",
    "urine_dipstick_clean_status",
    "pain_status",
    "breathalyzer_status",
    "hemocue_status",
]

REG_FEATURES_CATEG_WITH_TRIAGE = ["sex", "transport_grouped", "age_group", "complaint_category_reg", "triage"]
REG_FEATURES_CATEG_NO_TRIAGE   = ["sex", "transport_grouped", "age_group", "complaint_category_reg"]

ALL_REG_FEATURES_WITH_TRIAGE = REG_FEATURES_CATEG_WITH_TRIAGE + REG_FEATURES_STATUS
ALL_REG_FEATURES_NO_TRIAGE   = REG_FEATURES_CATEG_NO_TRIAGE   + REG_FEATURES_STATUS




Le cluster de référence doit être le plus grand ou le plus "basal" cliniquement — par exemple le cluster des patients jeunes, non urgents, faible consommation. Tous les OR s'interprètent par rapport à lui.
Avec N=120 000 tout sera significatif comme pour Cramér's V — donc ici aussi tu regardes la magnitude de l'OR, pas juste le p-value. Un OR de 1.05 n'est pas intéressant même si p < 0.001. Concentre-toi sur les OR > 1.5 ou < 0.67 (effet modéré).

In [146]:
# ==============================================================================
# CHUNK 2 — Reference categories + helper function
# Update REF_CATEGORIES after reading modalities from Chunk 1
# ==============================================================================

REF_CATEGORIES = {
    "sex"                         : "M",
    "transport_grouped"           : "Personal",
    "age_group"                   : "15-30",
    "complaint_category_reg"      : "Trauma",
    "triage"                      : "3",
    "bp_status"                   : "normotension",
    "hr_status"                   : "normocardia",
    "temp_status"                 : "normothermia",
    "sat_status"                  : "normal",
    "rr_status"                   : "normal",
    "o2_flow_status"              : "off",
    "gcs_status"                  : "normal",
    "cap_blood_sugar_status"      : "normoglycemia",
    "anisocoria_status"           : "no",
    "urine_dipstick_clean_status" : "negative",
    "pain_status"                 : "no_pain",
    "breathalyzer_status"         : "negative",
    "hemocue_status"              : "normal",
}



# ── Helper — format results table ──────────────────────────────────────────────

def format_mnlogit_results(result, ref_cluster, CLUSTER_LABELS):
    rows = []
    conf = result.conf_int()

    param_cols_no_ref = [c for c in result.params.columns if c != ref_cluster]
    conf_keys         = conf.index.get_level_values(0).unique().tolist()

    for col_idx, cluster in enumerate(param_cols_no_ref):
        params     = result.params[cluster]
        pvalues    = result.pvalues[cluster]
        conf_key   = conf_keys[col_idx]
        conf_clust = conf.xs(conf_key)

        if col_idx == 0:
            print(f"--- cluster {cluster} ---")
            print("params index:", params.index.tolist()[:5])
            print("conf_clust index:", conf_clust.index.tolist()[:5])


        for var in params.index:
            if var == "Intercept":
                continue

            or_val = np.exp(params[var])
            pval   = pvalues[var]

            try:
                row     = conf_clust.loc[var]
                ci_low  = np.exp(row["lower"])
                ci_high = np.exp(row["upper"])
            except KeyError:
                idx     = list(params.index).index(var)
                ci_low  = np.exp(conf_clust.iloc[idx]["lower"])
                ci_high = np.exp(conf_clust.iloc[idx]["upper"])

            clean_var = re.sub(
                r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
                r"\1 = \2", var
            )

            rows.append({
                "cluster_vs_ref": f"{CLUSTER_LABELS.get(cluster, f'C{cluster}')} vs {CLUSTER_LABELS.get(ref_cluster, f'C{ref_cluster}')}",
                "variable"      : clean_var,
                "OR"            : round(or_val,  3),
                "CI_low"        : round(ci_low,  3),
                "CI_high"       : round(ci_high, 3),
                "p_value"       : round(pval,    4),
                "significant"   : pval < 0.05,
            })
    return pd.DataFrame(rows)


print("Reference categories set:")
for k, v in REF_CATEGORIES.items():
    print(f"  {k:<35} ref = {v}")
print("\nHelper function loaded.")

Reference categories set:
  sex                                 ref = M
  transport_grouped                   ref = Personal
  age_group                           ref = 15-30
  complaint_category_reg              ref = Trauma
  triage                              ref = 3
  bp_status                           ref = normotension
  hr_status                           ref = normocardia
  temp_status                         ref = normothermia
  sat_status                          ref = normal
  rr_status                           ref = normal
  o2_flow_status                      ref = off
  gcs_status                          ref = normal
  cap_blood_sugar_status              ref = normoglycemia
  anisocoria_status                   ref = no
  urine_dipstick_clean_status         ref = negative
  pain_status                         ref = no_pain
  breathalyzer_status                 ref = negative
  hemocue_status                      ref = normal

Helper function loaded.


In [140]:
# ==============================================================================
# CHUNK 3 — Splines — age vs cluster membership (multinomial vs ref clsuter)
# ==============================================================================

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']

    # ── Paths ─────────────────────────────────────────────────────────────────
    CSV_PATH    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    OUT_DIR_REG = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    os.makedirs(OUT_DIR_REG, exist_ok=True)

    # ── Load ──────────────────────────────────────────────────────────────────
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
    )
    df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
    df_clust['cluster_label'] = pd.Categorical(
        df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
    )

    # ── Build df_reg ──────────────────────────────────────────────────────────
    cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'cluster_label', 'age']
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])

    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)

    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print(f"\n{'='*60}")
    print(f"RUN — {n_clusters} clusters (mcs={mcs}, ms={ms})")
    print(f"{'='*60}")
    print(f"Regression dataset: {len(df_reg):,} patients")




    OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
    ref_c_spline = ref_c
    ref_label    = CLUSTER_LABELS[ref_c_spline]
    os.makedirs(OUT_DIR_SPLINE, exist_ok=True)



    df_spline = df_reg[["age", "cluster"]].dropna()
    df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

    # ── Cluster 3 (C5) = référence → en premier dans l'encodage ──────────────────
    df_spline["cluster"] = pd.Categorical(
        df_spline["cluster"],
        categories=[ref_c_spline] + sorted([c for c in df_spline["cluster"].unique() if c != ref_c_spline])
    )

    age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

    spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
    spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")

    spline_basis_model = sm.add_constant(spline_basis)
    spline_basis_pred  = sm.add_constant(spline_basis_range)

    # ── MNLogit ───────────────────────────────────────────────────────────────────
    y_cat            = df_spline["cluster"].cat.codes
    model            = sm.MNLogit(y_cat, spline_basis_model).fit(method="newton", maxiter=500, disp=False)
    # Trier non_ref_clusters selon CLUSTER_ORDER
    non_ref_clusters = [
        c for label in CLUSTER_ORDER
        for c, lbl in CLUSTER_LABELS.items()
        if lbl == label and c != ref_c_spline and c in df_spline["cluster"].unique()
    ]
    n_outcomes       = len(non_ref_clusters)
    logit_preds      = spline_basis_pred.values @ model.params.values

    # ── IC ────────────────────────────────────────────────────────────────────────
    cov      = model.cov_params()
    n_params = spline_basis_pred.shape[1]
    ci_lows, ci_highs = [], []

    for k in range(n_outcomes):
        idx   = slice(k * n_params, (k + 1) * n_params)
        cov_k = cov.values[idx, idx]
        grad  = spline_basis_pred.values
        se_k  = np.sqrt((grad @ cov_k @ grad.T).diagonal())
        ci_lows.append(logit_preds[:, k] - 1.96 * se_k)
        ci_highs.append(logit_preds[:, k] + 1.96 * se_k)

    # ── FIGURE 1 — Un subplot par cluster non-référence ──────────────────────────
    palette = sns.color_palette("tab10", n_outcomes)
    n_cols  = min(3, n_outcomes)
    n_rows  = (n_outcomes + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 6, n_rows * 5), facecolor="white")
    fig.patch.set_facecolor("white")
    axes = np.array(axes).flatten()

    df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

    for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
        ax        = axes[i]
        c_label   = CLUSTER_LABELS.get(c, f"Cluster {c}")

        # ── Points bruts cohérents avec MNLogit (c vs ref uniquement) ────────────
        df_pair = df_spline[df_spline["cluster"].isin([c, ref_c_spline])]
        raw = (
            df_pair.groupby("age_bin", observed=True)
            .apply(lambda x: (x["cluster"] == c).mean())
            .reset_index()
        )
        raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
        raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
        raw = raw[(raw[0] > 0) & (raw[0] < 1)]

        ax.scatter(raw["age_mid"], raw["logit_raw"],
                   color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
        ax.plot(age_range, logit_preds[:, i],
                color=color, linewidth=2, label="Spline fit")
        ax.fill_between(age_range, ci_lows[i], ci_highs[i],
                        alpha=0.2, color=color, label="95% CI")

        for boundary in [30, 45, 60, 75]:
            ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

        ax.set_facecolor("white")
        ax.set_title(f"{c_label}\nvs {ref_label} (ref)",
                     fontsize=10, fontweight="bold", color="black")
        ax.set_xlabel("Age", fontsize=10, color="black")
        ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=9, color="black")
        ax.tick_params(colors="black")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    legend_elements = [
        Line2D([0], [0], color="grey",      linewidth=0, marker="o",
               markersize=6, alpha=0.5, label="Raw proportion (logit)"),
        Line2D([0], [0], color="black",     linewidth=2,  label="Spline fit"),
        Line2D([0], [0], color="black",     linewidth=8,  alpha=0.2, label="95% CI"),
        Line2D([0], [0], color="lightgrey", linewidth=1,  linestyle="--", label="Age group boundary"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4,
               fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
    fig.suptitle(
        f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
        f"Dashed lines = age_group boundaries (30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black",
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_mnlogit_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_by_cluster_mnlogit_{n_clusters}clusters.png")

    # ── FIGURE 2 — Tous les clusters sur un seul graphe ──────────────────────
    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")

    for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
        c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
        ax.plot(age_range, logit_preds[:, i], color=color, linewidth=2, label=c_label)

    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ymax = ax.get_ylim()[1]
    for boundary in [30, 45, 60, 75]:
        ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
                fontsize=8, color="grey", va="top")

    ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Age (years)", fontsize=12, color="black")
    ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=12, color="black")
    ax.set_title(
        f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
        f"(dashed lines = age_group boundaries: 30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black", pad=15,
    )
    ax.tick_params(colors="black")
    ax.legend(title=f"Cluster (vs {ref_label})", bbox_to_anchor=(1.02, 1),
                loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_mnlogit_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_all_clusters_mnlogit_{n_clusters}clusters.png")


RUN — 5 clusters (mcs=3407, ms=170)
Regression dataset: 56,784 patients
Saved: spline_age_by_cluster_mnlogit_5clusters.png
Saved: spline_age_all_clusters_mnlogit_5clusters.png

RUN — 9 clusters (mcs=2271, ms=15)
Regression dataset: 56,784 patients
Saved: spline_age_by_cluster_mnlogit_9clusters.png
Saved: spline_age_all_clusters_mnlogit_9clusters.png


In [141]:
# ==============================================================================
# CHUNK 3bis — Splines binaires — cluster k vs the rest
# ==============================================================================

for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']

    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"
    OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
    os.makedirs(OUT_DIR_SPLINE, exist_ok=True)

    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
    )
    cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'age']
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=['cluster', 'age'])
    df_reg['cluster'] = df_reg['cluster'].astype(int)

    df_spline = df_reg[["age", "cluster"]].dropna()
    df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

    clusters = [
        c for label in CLUSTER_ORDER
        for c, lbl in CLUSTER_LABELS.items()
        if lbl == label and c in df_spline["cluster"].unique()
    ]
    age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

    spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
    spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")

    # ── FIGURE 1 — One subplot per cluster ───────────────────────────────────
    n_cols = min(3, len(clusters))
    n_rows = (len(clusters) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 6, n_rows * 5), facecolor="white")
    fig.patch.set_facecolor("white")
    axes = np.array(axes).flatten()

    df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

    for i, c in enumerate(clusters):
        ax      = axes[i]
        c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
        y_bin   = (df_spline["cluster"] == c).astype(int)
        model   = sm.Logit(y_bin, spline_basis).fit(disp=False)

        logit_pred = spline_basis_range.values @ model.params.values
        cov        = model.cov_params()
        gradient   = np.array(spline_basis_range)
        se         = np.sqrt((gradient @ cov.values @ gradient.T).diagonal())
        ci_low     = logit_pred - 1.96 * se
        ci_high    = logit_pred + 1.96 * se

        raw = (
            df_spline.groupby("age_bin", observed=True)
            .apply(lambda x: (x["cluster"] == c).mean())
            .reset_index()
        )
        raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
        raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
        raw = raw[(raw[0] > 0) & (raw[0] < 1)]

        ax.scatter(raw["age_mid"], raw["logit_raw"],
                   color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
        ax.plot(age_range, logit_pred,
                color="steelblue", linewidth=2, label="Spline fit (logit)")
        ax.fill_between(age_range, ci_low, ci_high,
                        alpha=0.2, color="steelblue", label="95% CI")
        for boundary in [30, 45, 60, 75]:
            ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

        ax.set_facecolor("white")
        ax.set_title(c_label, fontsize=10, fontweight="bold", color="black")
        ax.set_xlabel("Age",  fontsize=10, color="black")
        ax.set_ylabel("Log-odds (logit)", fontsize=10, color="black")
        ax.tick_params(colors="black")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    legend_elements = [
        Line2D([0], [0], color="grey",      linewidth=0, marker="o",
               markersize=6, alpha=0.5, label="Raw proportion"),
        Line2D([0], [0], color="steelblue", linewidth=2, label="Spline fit"),
        Line2D([0], [0], color="steelblue", linewidth=8, alpha=0.2, label="95% CI"),
        Line2D([0], [0], color="lightgrey", linewidth=1, linestyle="--",
               label="Age group boundary"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4,
               fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
    fig.suptitle(
        f"Spline fit — P(cluster membership | age) — {n_clusters} clusters\n"
        "Dashed lines = age_group boundaries (30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black",
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_by_cluster_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_by_cluster_{n_clusters}clusters.png")

    # ── FIGURE 2 — All clusters on one figure ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    palette = sns.color_palette("tab10", len(clusters))

    for c, color in zip(clusters, palette):
        c_label    = CLUSTER_LABELS.get(c, f"Cluster {c}")
        y_bin      = (df_spline["cluster"] == c).astype(int)
        model      = sm.Logit(y_bin, spline_basis).fit(disp=False)
        logit_pred = spline_basis_range.values @ model.params.values
        ax.plot(age_range, logit_pred, color=color, linewidth=2, label=c_label)

    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ymax = ax.get_ylim()[1]
    for boundary in [30, 45, 60, 75]:
        ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
                fontsize=8, color="grey", va="top")

    ax.set_xlabel("Age (years)",      fontsize=12, color="black")
    ax.set_ylabel("Log-odds (logit)", fontsize=12, color="black")
    ax.set_title(
        f"Spline fit — Log-odds of cluster membership by age — {n_clusters} clusters\n"
        "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
        fontsize=13, fontweight="bold", color="black", pad=15,
    )
    ax.tick_params(colors="black")
    ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1),
              loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_SPLINE, f"spline_age_all_clusters_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: spline_age_all_clusters_{n_clusters}clusters.png")

Saved: spline_age_by_cluster_5clusters.png
Saved: spline_age_all_clusters_5clusters.png
Saved: spline_age_by_cluster_9clusters.png
Saved: spline_age_all_clusters_9clusters.png



Cluster 4 (brun) — forte décroissance avec l'âge
C'est ton cluster "passage simple / aucun examen". Les jeunes (15-30 ans) y sont massivement représentés (~30%) et cette probabilité chute fortement après 45 ans. Les jeunes adultes consultent aux urgences pour des motifs simples ne nécessitant pas d'explorations.
Cluster 7 (jaune-vert) — décroissance progressive
Surreprésenté chez les jeunes (~20%) et quasi absent après 75 ans. Probablement lié à des motifs traumatologiques ou musculo-squelettiques, typiques des jeunes actifs.
Cluster 5 (rose) — forte croissance avec l'âge
Probabilité qui monte fortement après 60 ans et explose après 75 ans (~30%). C'est ton cluster de prise en charge lourde des patients âgés.
Cluster 3 (violet) — légère croissance
Stable avec une légère augmentation après 60 ans — probablement les bilans cardiovasculaires/neurologiques plus fréquents avec l'âge.
Clusters 0 et 1 (orange, vert) — croissance modérée
Augmentent progressivement avec l'âge, pic vers 70-80 ans puis redescendent légèrement. Bilans biologiques et imagerie plus fréquents chez les patients d'âge moyen à âgés.
Cluster 6 (gris) — en cloche
Pic vers 70-80 ans puis décroissance — profil typique des patients âgés mais pas très vieux.
Cluster 2 (rouge) et Outliers (bleu) — en cloche tardive
Pic vers 70-80 ans, profils atypiques ou complexes plus fréquents chez les personnes âgées.
Cluster 8 (cyan) — décroissance
Surreprésenté chez les jeunes, décroît avec l'âge.

Ce que ça justifie pour ta régression :
Les courbes sont clairement non-linéaires — elles montent, descendent, ont des inflexions — ce qui justifie pleinement age_group en catégoriel plutôt qu'age en continu. Et les inflexions coïncident globalement avec tes frontières à 30, 45, 60 et 75 ans, ce qui valide tes coupures a priori.

In [147]:
# ==============================================================================
# CHUNK 4 — Univariate multinomial logistic regressions
# ==============================================================================


for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"

    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
    )
    df_clust['cluster_label'] = df_clust['cluster'].map(CLUSTER_LABELS)
    df_clust['cluster_label'] = pd.Categorical(
        df_clust['cluster_label'], categories=CLUSTER_ORDER, ordered=True
    )
    cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'cluster_label', 'age']
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print("\n" + "="*60)
    print("UNIVARIATE REGRESSIONS")
    print("="*60)

    all_uni_res = []

    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            print(f"MISSING  {var}")
            continue

        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula = f"cluster ~ C({var}, Treatment('{ref_val}'))"
        print(f"Trying: {formula}")

        model = None
        for method in ["bfgs", "lbfgs", "cg", "newton"]:
            try:
                model = mnlogit(formula, data=df_reg).fit(
                    method  = method,
                    maxiter = 2000,
                    gtol    = 1e-5,
                    disp    = False,
                )
                if not model.mle_retvals.get("converged", True):
                    print(f"WARNING {var} (method={method}): converged=False")
                else:
                    print(f"OK  {var} (method={method})")
                break
            except Exception as e:
                print(f"  {method} failed: {e}")

        if model is None:
            print(f"FAILED  {var} — all methods failed")
            continue

        df_res = format_mnlogit_results(model, ref_c, CLUSTER_LABELS)
        df_res["model"]   = "univariate"
        df_res["feature"] = var
        all_uni_res.append(df_res)

    # ── Concat + CSV ──────────────────────────────────────────────────────────────
    if len(all_uni_res) == 0:
        print("WARNING: aucun modèle n'a convergé")
    else:
        df_univariate = pd.concat(all_uni_res, ignore_index=True)
        df_univariate = df_univariate.sort_values(
            ["cluster_vs_ref", "feature"]
        ).reset_index(drop=True)

        df_univariate.to_csv(os.path.join(OUT_DIR_REG, f"univariate_results_{n_clusters}clusters.csv"), index=False)
        print(f"\nSaved: univariate_results.csv ({len(df_univariate)} rows)")

        mask = (df_univariate["CI_low"] > df_univariate["OR"]) | (df_univariate["CI_high"] < df_univariate["OR"])
        print(df_univariate[mask][["cluster_vs_ref", "variable", "OR", "CI_low", "CI_high", "p_value"]])

        # ── Forest plot — un plot par cluster_vs_ref ──────────────────────────────
        ref_label       = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
        clusters_vs_ref = sorted(df_univariate["cluster_vs_ref"].unique())

        for clust_label in clusters_vs_ref:
            df_plot = (
                df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]

            fig, ax = plt.subplots(
                figsize   = (10, max(8, n_vars * 0.35)),
                facecolor = "white"
            )
            ax.set_facecolor("white")
            plt.rcParams['text.color'] = 'black'
            plt.rcParams['axes.labelcolor'] = 'black'
            plt.rcParams['xtick.color'] = 'black'
            plt.rcParams['ytick.color'] = 'black'
            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"],
                np.log(df_plot["OR"]),
                xerr   = [err_low, err_high],
                color   = colors,
                alpha   = 0.8,
                capsize = 3,
                ecolor  = "grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=12)
            ax.set_title(
                f"Univariate OR — {clust_label}\n"
                f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)\n"
                f"ref = {ref_label}",
                fontsize=11, fontweight="bold", color="black"
            )
            ax.tick_params(axis="y", labelsize=9)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            plt.tight_layout()
            fname_clean = (
                clust_label
                .replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_univariate_{fname_clean}_{n_clusters}clusters.png"
            plt.savefig(os.path.join(OUT_DIR_REG, fname),
                        dpi=200, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"Saved: {fname}")

        # ── Forest plot combiné — tous les clusters sur une seule figure ──────
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 10, n_rows_fp * max(8, n_vars * 0.35))
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = (
                df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars_i = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"], np.log(df_plot["OR"]),
                xerr=[err_low, err_high],
                color=colors, alpha=0.8, capsize=3, ecolor="grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=10)
            ax.set_title(clust_label, fontsize=9, fontweight="bold")
            ax.tick_params(axis="y", labelsize=7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"Univariate OR — all clusters vs {ref_label} — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_REG, f"forest_plot_univariate_all_clusters_{n_clusters}clusters.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_univariate_all_clusters_{n_clusters}clusters.png")


UNIVARIATE REGRESSIONS
Trying: cluster ~ C(sex, Treatment('M'))
OK  sex (method=bfgs)
--- cluster 0 ---
params index: ['Intercept', "C(sex, Treatment('M'))[T.F]"]
conf_clust index: ['Intercept', "C(sex, Treatment('M'))[T.F]"]
Trying: cluster ~ C(transport_grouped, Treatment('Personal'))
OK  transport_grouped (method=bfgs)
--- cluster 0 ---
params index: ['Intercept', "C(transport_grouped, Treatment('Personal'))[T.Ambulance]", "C(transport_grouped, Treatment('Personal'))[T.Emergency services]", "C(transport_grouped, Treatment('Personal'))[T.Post medical advice]", "C(transport_grouped, Treatment('Personal'))[T.Unknown]"]
conf_clust index: ['Intercept', "C(transport_grouped, Treatment('Personal'))[T.Ambulance]", "C(transport_grouped, Treatment('Personal'))[T.Emergency services]", "C(transport_grouped, Treatment('Personal'))[T.Post medical advice]", "C(transport_grouped, Treatment('Personal'))[T.Unknown]"]
Trying: cluster ~ C(age_group, Treatment('15-30'))
OK  age_group (method=bfgs)
--- 

In [143]:
# ==============================================================================
# CHUNK 4b — Univariate binary logistic regressions — Outliers (-1) vs rest
# ==============================================================================
for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"

    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
    )
    cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'age']
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)

    for col in REG_FEATURES_STATUS:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)

    print("\n" + "="*60)
    print("UNIVARIATE REGRESSIONS — OUTLIERS vs REST")
    print("="*60)

    # Variable binaire : outlier (-1) = 1, tout le reste = 0
    print(f"Outliers: {df_reg['is_outlier'].sum()} / {len(df_reg)}")

    all_uni_res_outlier = []

    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            print(f"MISSING  {var}")
            continue

        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula = f"is_outlier ~ C({var}, Treatment('{ref_val}'))"
        print(f"Trying: {formula}")

        model = None
        for method in ["bfgs", "lbfgs", "cg", "newton"]:
            try:
                model = smf.logit(formula, data=df_reg).fit(
                    method  = method,
                    maxiter = 2000,
                    gtol    = 1e-5,
                    disp    = False,
                )
                if not model.mle_retvals.get("converged", True):
                    print(f"WARNING {var} (method={method}): converged=False, résultats conservés")
                else:
                    print(f"OK  {var} (method={method})")
                break
            except Exception as e:
                print(f"  {method} failed: {e}")

        if model is None:
            print(f"FAILED  {var} — all methods failed")
            continue

        # ── Extraire OR, IC, p-value ──────────────────────────────────────────────
        params = model.params
        conf   = model.conf_int()
        pvals  = model.pvalues
        conf.columns = ["CI_low", "CI_high"]

        rows = []
        for v in params.index:
            if v == "Intercept":
                continue

            clean_var = re.sub(
                r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
                r"\1 = \2", v
            )
            rows.append({
                "variable"   : clean_var,
                "OR"         : round(np.exp(params[v]),       3),
                "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]),  3),
                "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
                "p_value"    : round(pvals[v], 4),
                "significant": pvals[v] < 0.05,
                "feature"    : var,
                "model"      : "univariate_outlier",
            })

        all_uni_res_outlier.append(pd.DataFrame(rows))

    # ── Concat ────────────────────────────────────────────────────────────────────
    if len(all_uni_res_outlier) == 0:
        print("WARNING: aucun modèle n'a convergé")
    else:
        df_outlier = pd.concat(all_uni_res_outlier, ignore_index=True)
        df_outlier = df_outlier.sort_values(["feature", "variable"]).reset_index(drop=True)

        df_outlier.to_csv(os.path.join(OUT_DIR_REG,  f"univariate_outlier_results_{n_clusters}clusters.csv"), index=False)
        print(f"\nSaved: univariate_outlier_results.csv ({len(df_outlier)} rows)")

    # ── Forest plot outliers ──────────────────────────────────────────────────────
    n_vars  = len(df_outlier)
    df_plot = df_outlier.sort_values("OR", ascending=True).reset_index(drop=True)

    # Label = feature + variable pour bien distinguer
    df_plot["label"] = df_plot["feature"] + " — " + df_plot["variable"]

    colors = [
        "crimson"   if (s and or_val > 1)  else
        "steelblue" if (s and or_val <= 1) else
        "lightgrey"
        for s, or_val in zip(df_plot["significant"], df_plot["OR"])
    ]

    fig, ax = plt.subplots(
        figsize   = (10, max(8, n_vars * 0.35)),
        facecolor = "white"
    )
    ax.set_facecolor("white")

    err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
    err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
    ax.barh(
        df_plot["variable"],
        np.log(df_plot["OR"]),
        xerr   = [err_low, err_high],
        color   = colors,
        alpha   = 0.8,
        capsize = 3,
        ecolor  = "grey",
    )

    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("log(OR) = β", fontsize=12)
    ax.set_title(
        "Univariate OR — External variables → Outliers vs rest\n"
        "(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
        fontsize=13, fontweight="bold", color="black"
    )
    ax.tick_params(axis="y", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_REG, f"forest_plot_univariate_outliers_{n_clusters}clusters.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print("Saved: forest_plot_univariate_outliers.png")


UNIVARIATE REGRESSIONS — OUTLIERS vs REST
Outliers: 1898 / 56784
Trying: is_outlier ~ C(sex, Treatment('M'))
OK  sex (method=bfgs)
Trying: is_outlier ~ C(transport_grouped, Treatment('Personal'))
OK  transport_grouped (method=bfgs)
Trying: is_outlier ~ C(age_group, Treatment('15-30'))
OK  age_group (method=bfgs)
Trying: is_outlier ~ C(complaint_category_reg, Treatment('Trauma'))
OK  complaint_category_reg (method=bfgs)
Trying: is_outlier ~ C(triage, Treatment('3'))
OK  triage (method=bfgs)
Trying: is_outlier ~ C(bp_status, Treatment('normotension'))
OK  bp_status (method=bfgs)
Trying: is_outlier ~ C(hr_status, Treatment('normocardia'))
OK  hr_status (method=bfgs)
Trying: is_outlier ~ C(temp_status, Treatment('normothermia'))
OK  temp_status (method=bfgs)
Trying: is_outlier ~ C(sat_status, Treatment('normal'))
OK  sat_status (method=bfgs)
Trying: is_outlier ~ C(rr_status, Treatment('normal'))
OK  rr_status (method=bfgs)
Trying: is_outlier ~ C(o2_flow_status, Treatment('off'))
OK  o2_fl

In [144]:
pd.crosstab(df_reg["triage"], df_reg["cluster"])

cluster,-1,0,1,2,3,4,5,6,7,8
triage,,,,,,,,,,
1,49,86,3,0,1,27,7,6,6,12
2,1628,5145,897,440,214,2349,2124,1204,761,1020
3,1856,4125,1944,2974,1619,2269,2370,964,1324,1412
4,1031,890,1225,6598,3480,746,832,255,733,379
5,131,60,217,2497,681,31,46,12,85,19


In [145]:
# ==============================================================================
# CHUNK 5 — Multivariate multinomial logistic regression
# ==============================================================================
for config in CLUSTERING_RUNS:
    n_clusters     = config['n_clusters']
    mcs            = config['mcs']
    ms             = config['ms']
    ref_c          = config['ref_cluster']
    CLUSTER_LABELS = config['cluster_labels']
    CLUSTER_ORDER  = config['cluster_order']
    OUT_DIR_REG    = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/logistic_regression"

    CSV_PATH = f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv"
    df_clust = pd.read_csv(CSV_PATH, low_memory=False)
    df_clust['complaint_category_reg'] = df_clust['complaint_category'].replace(
        {cat: 'Metabolic_Hematologic_Toxic' for cat in SYSTEMIC_RARE}
    )
    cols_needed = ALL_REG_FEATURES_WITH_TRIAGE + ['cluster', 'cluster_label', 'age']
    df_reg = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
    df_reg = df_reg.dropna(subset=ALL_REG_FEATURES_WITH_TRIAGE + ['cluster'])
    df_reg['cluster']                = df_reg['cluster'].astype(int)
    df_reg['triage']                 = df_reg['triage'].astype(float).astype(int).astype(str)
    df_reg['age_group']              = df_reg['age_group'].astype(str)
    df_reg['sex']                    = df_reg['sex'].astype(str)
    df_reg['transport_grouped']      = df_reg['transport_grouped'].astype(str)
    df_reg['complaint_category_reg'] = df_reg['complaint_category_reg'].astype(str)
    for col in REG_FEATURES_STATUS:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].astype(str)
    df_reg['is_outlier'] = (df_reg['cluster'] == -1).astype(int)
    df_clusters_only = df_reg[df_reg['cluster'] != -1].copy()

    print("\n" + "="*60)
    print("MULTIVARIATE REGRESSION")
    print("="*60)

    formula_parts = []
    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula_parts.append(f"C({var}, Treatment('{ref_val}'))")

    formula_multi = "cluster ~ " + " + ".join(formula_parts)
    print(f"\nFormula:\n{formula_multi}\n")

    # ── Fit avec fallback comme l'univarié ────────────────────────────────────────
    model_multi = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_multi = mnlogit(formula_multi, data=df_clusters_only).fit(
                method  = method,
                maxiter = 2000,
                gtol    = 1e-5,
                disp    = False,
            )
            if not model_multi.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False, résultats conservés")
            else:
                print(f"OK multivariate (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_multi is None:
        print("FAILED — all methods failed")
    else:
        # ── Extraire résultats via format_mnlogit_results ─────────────────────────
        df_multivariate = format_mnlogit_results(model_multi, ref_c, CLUSTER_LABELS)
        df_multivariate["model"] = "multivariate"

        df_multivariate = df_multivariate.sort_values(
            ["cluster_vs_ref", "variable"]
        ).reset_index(drop=True)

        df_multivariate.to_csv(
            os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_with_triage.csv"), index=False
        )
        print(f"\nSaved: multivariate_results.csv ({len(df_multivariate)} rows)")

        # ── Forest plot — une seule colonne par cluster_vs_ref ────────────────────
        clusters_vs_ref = sorted(df_multivariate["cluster_vs_ref"].unique())

        ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")

        for clust_label in clusters_vs_ref:
            df_plot = (
                df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]

            fig, ax = plt.subplots(
                figsize   = (10, max(8, n_vars * 0.35)),
                facecolor = "white"
            )
            ax.set_facecolor("white")

            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"],
                np.log(df_plot["OR"]),
                xerr   = [err_low, err_high],
                color   = colors,
                alpha   = 0.8,
                capsize = 3,
                ecolor  = "grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=12)
            ax.set_title(
                f"Multivariate OR — {clust_label}\n"
                f"(crimson = significant OR>1 | steelblue = significant OR<1 | grey = ns)\n"
                f"ref = {ref_label}",
                fontsize=11, fontweight="bold", color="black"
            )
            ax.tick_params(axis="y", labelsize=9)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            plt.tight_layout()

            # ── Nom de fichier nettoyé ────────────────────────────────────────────────
            fname_clean = (
                clust_label
                .replace(" ", "_")
                .replace("—", "-")
                .replace("+", "plus")
                .replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_with_triage.png"
            plt.savefig(
                os.path.join(OUT_DIR_REG, fname),
                dpi=200, bbox_inches="tight", facecolor="white"
            )
            plt.close()
            print(f"Saved: {fname}")

        # ── Forest plot combiné with triage ───────────────────────────────────
        n_clusters_plot = len(clusters_vs_ref)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 10, n_rows_fp * max(8, n_vars * 0.35))
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref):
            ax = axes[ax_i]
            df_plot = (
                df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"], np.log(df_plot["OR"]),
                xerr=[err_low, err_high],
                color=colors, alpha=0.8, capsize=3, ecolor="grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=10)
            ax.set_title(clust_label, fontsize=9, fontweight="bold")
            ax.tick_params(axis="y", labelsize=7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"Multivariate OR (with triage) — all clusters vs {ref_label} — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_with_triage.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_multivariate_all_clusters_{n_clusters}clusters_with_triage.png")


    # ── MULTIVARIATE — SANS TRIAGE ────────────────────────────────────────────
    print("\n── Multivariate sans triage ──")
    formula_parts_nt = []
    for var in ALL_REG_FEATURES_NO_TRIAGE:
        if var not in df_clusters_only.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_clusters_only[var].mode()[0])
        formula_parts_nt.append(f"C({var}, Treatment('{ref_val}'))")

    formula_multi_nt = "cluster ~ " + " + ".join(formula_parts_nt)

    model_multi_nt = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_multi_nt = mnlogit(formula_multi_nt, data=df_clusters_only).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_multi_nt.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate no triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_multi_nt is None:
        print("FAILED — all methods failed")
    else:
        df_multi_nt = format_mnlogit_results(model_multi_nt, ref_c, CLUSTER_LABELS)
        df_multi_nt["model"] = "multivariate_no_triage"
        df_multi_nt = df_multi_nt.sort_values(["cluster_vs_ref", "variable"]).reset_index(drop=True)
        df_multi_nt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_results_{n_clusters}clusters_no_triage.csv"), index=False)
        print(f"Saved: multivariate_results_{n_clusters}clusters_no_triage.csv")

        ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
        for clust_label in sorted(df_multi_nt["cluster_vs_ref"].unique()):
            df_plot = (
                df_multi_nt[df_multi_nt["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True).reset_index(drop=True)
            )
            n_vars = len(df_plot)
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)), facecolor="white")
            ax.set_facecolor("white")
            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"],
                np.log(df_plot["OR"]),
                xerr   = [err_low, err_high],
                color   = colors,
                alpha   = 0.8,
                capsize = 3,
                ecolor  = "grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=12)
            ax.set_title(
                f"Multivariate OR (no triage) — {clust_label}\n"
                f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)\nref = {ref_label}",
                fontsize=11, fontweight="bold", color="black"
            )
            ax.tick_params(axis="y", labelsize=9)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.tight_layout()
            fname_clean = (
                clust_label.replace(" ", "_").replace("—", "-")
                .replace("+", "plus").replace("/", "-")
                .replace("(", "").replace(")", "")
            )
            fname = f"forest_plot_multivariate_{fname_clean}_{n_clusters}clusters_no_triage.png"
            plt.savefig(os.path.join(OUT_DIR_REG, fname), dpi=200, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"Saved: {fname}")

        # ── Forest plot combiné no triage ─────────────────────────────────────
        clusters_vs_ref_nt = sorted(df_multi_nt["cluster_vs_ref"].unique())
        n_clusters_plot = len(clusters_vs_ref_nt)
        n_cols_fp = min(3, n_clusters_plot)
        n_rows_fp = (n_clusters_plot + n_cols_fp - 1) // n_cols_fp

        fig, axes = plt.subplots(
            n_rows_fp, n_cols_fp,
            figsize=(n_cols_fp * 10, n_rows_fp * max(8, n_vars * 0.35))
        )
        axes = np.array(axes).flatten()

        for ax_i, clust_label in enumerate(clusters_vs_ref_nt):
            ax = axes[ax_i]
            df_plot = (
                df_multi_nt[df_multi_nt["cluster_vs_ref"] == clust_label]
                .sort_values("OR", ascending=True)
                .reset_index(drop=True)
            )
            colors = [
                "crimson"   if (s and or_val > 1)  else
                "steelblue" if (s and or_val <= 1) else
                "lightgrey"
                for s, or_val in zip(df_plot["significant"], df_plot["OR"])
            ]
            err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
            err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
            ax.barh(
                df_plot["variable"], np.log(df_plot["OR"]),
                xerr=[err_low, err_high],
                color=colors, alpha=0.8, capsize=3, ecolor="grey",
            )
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel("log(OR) = β", fontsize=10)
            ax.set_title(clust_label, fontsize=9, fontweight="bold")
            ax.tick_params(axis="y", labelsize=7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(ax_i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            f"Multivariate OR (no triage) — all clusters vs {ref_label} — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_all_clusters_{n_clusters}clusters_no_triage.png"),
            dpi=200, bbox_inches="tight"
        )
        plt.close()
        print(f"Saved: forest_plot_multivariate_all_clusters_{n_clusters}clusters_no_triage.png")


    # ── MULTIVARIATE OUTLIERS — with triage ───────────────────────────────────
    print("\n── Multivariate outliers avec triage ──")
    formula_parts_out = []
    for var in ALL_REG_FEATURES_WITH_TRIAGE:
        if var not in df_reg.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula_parts_out.append(f"C({var}, Treatment('{ref_val}'))")

    formula_out = "is_outlier ~ " + " + ".join(formula_parts_out)

    model_out = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_out = smf.logit(formula_out, data=df_reg).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_out.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate outliers with triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_out is None:
        print("FAILED — all methods failed")
    else:
        params = model_out.params
        conf   = model_out.conf_int()
        pvals  = model_out.pvalues
        conf.columns = ["CI_low", "CI_high"]
        rows = []
        for v in params.index:
            if v == "Intercept":
                continue
            clean_var = re.sub(r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]", r"\1 = \2", v)
            rows.append({
                "variable"   : clean_var,
                "OR"         : round(np.exp(params[v]), 3),
                "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]), 3),
                "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
                "p_value"    : round(pvals[v], 4),
                "significant": pvals[v] < 0.05,
            })
        df_out_multi_wt = pd.DataFrame(rows)
        df_out_multi_wt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_with_triage.csv"), index=False)

        df_plot = df_out_multi_wt.sort_values("OR", ascending=True).reset_index(drop=True)
        n_vars  = len(df_plot)
        colors  = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]
        err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
        err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
        fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)))
        ax.barh(df_plot["variable"], np.log(df_plot["OR"]),
                xerr=[err_low, err_high], color=colors, alpha=0.8, capsize=3, ecolor="grey")
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("log(OR) = β", fontsize=12)
        ax.set_title(
            f"Multivariate OR (with triage) — Outliers vs rest — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_outliers_{n_clusters}clusters_with_triage.png"),
                    dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved: forest_plot_multivariate_outliers_{n_clusters}clusters_with_triage.png")

    # ── MULTIVARIATE OUTLIERS — no triage ─────────────────────────────────────
    print("\n── Multivariate outliers sans triage ──")
    formula_parts_out_nt = []
    for var in ALL_REG_FEATURES_NO_TRIAGE:
        if var not in df_reg.columns:
            continue
        ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
        formula_parts_out_nt.append(f"C({var}, Treatment('{ref_val}'))")

    formula_out_nt = "is_outlier ~ " + " + ".join(formula_parts_out_nt)

    model_out_nt = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model_out_nt = smf.logit(formula_out_nt, data=df_reg).fit(
                method=method, maxiter=2000, gtol=1e-5, disp=False,
            )
            if not model_out_nt.mle_retvals.get("converged", True):
                print(f"WARNING (method={method}): converged=False")
            else:
                print(f"OK multivariate outliers no triage (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model_out_nt is None:
        print("FAILED — all methods failed")
    else:
        params = model_out_nt.params
        conf   = model_out_nt.conf_int()
        pvals  = model_out_nt.pvalues
        conf.columns = ["CI_low", "CI_high"]
        rows = []
        for v in params.index:
            if v == "Intercept":
                continue
            clean_var = re.sub(r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]", r"\1 = \2", v)
            rows.append({
                "variable"   : clean_var,
                "OR"         : round(np.exp(params[v]), 3),
                "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]), 3),
                "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
                "p_value"    : round(pvals[v], 4),
                "significant": pvals[v] < 0.05,
            })
        df_out_multi_nt = pd.DataFrame(rows)
        df_out_multi_nt.to_csv(os.path.join(OUT_DIR_REG, f"multivariate_outlier_results_{n_clusters}clusters_no_triage.csv"), index=False)

        df_plot = df_out_multi_nt.sort_values("OR", ascending=True).reset_index(drop=True)
        n_vars  = len(df_plot)
        colors  = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]
        err_low  = np.clip(np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),  0, None)
        err_high = np.clip(np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]), 0, None)
        fig, ax = plt.subplots(figsize=(10, max(8, n_vars * 0.35)))
        ax.barh(df_plot["variable"], np.log(df_plot["OR"]),
                xerr=[err_low, err_high], color=colors, alpha=0.8, capsize=3, ecolor="grey")
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("log(OR) = β", fontsize=12)
        ax.set_title(
            f"Multivariate OR (no triage) — Outliers vs rest — {n_clusters} clusters\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
            fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_REG, f"forest_plot_multivariate_outliers_{n_clusters}clusters_no_triage.png"),
                    dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved: forest_plot_multivariate_outliers_{n_clusters}clusters_no_triage.png")


MULTIVARIATE REGRESSION

Formula:
cluster ~ C(sex, Treatment('M')) + C(transport_grouped, Treatment('Personal')) + C(age_group, Treatment('15-30')) + C(complaint_category_reg, Treatment('Trauma')) + C(triage, Treatment('3')) + C(bp_status, Treatment('normotension')) + C(hr_status, Treatment('normocardia')) + C(temp_status, Treatment('normothermia')) + C(sat_status, Treatment('normal')) + C(rr_status, Treatment('normal')) + C(o2_flow_status, Treatment('off')) + C(gcs_status, Treatment('normal')) + C(cap_blood_sugar_status, Treatment('normoglycemia')) + C(anisocoria_status, Treatment('no')) + C(urine_dipstick_clean_status, Treatment('negative')) + C(pain_status, Treatment('no_pain')) + C(breathalyzer_status, Treatment('negative')) + C(hemocue_status, Treatment('normal'))

OK multivariate (method=bfgs)

Saved: multivariate_results.csv (189 rows)
Saved: forest_plot_multivariate_C1_-_UHCD_plus_hospitalization_plus_heavy_workup_vs_C5_-_Discharged_plus_minimal_consumption_5clusters_with_tria